# 📶 Project 12.3 — Signal Sequence Analyser
**DSA for Mechatronics · Week 12 — Dynamic Programming**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, time
from functools import lru_cache
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
Two sensor channels emit discrete signal codes. We analyse their relationship
using three classic sequence DP algorithms:

1. **Longest Common Subsequence (LCS)** — find the longest subsequence shared
   by both channels (not necessarily contiguous)
2. **Longest Increasing Subsequence (LIS)** — find the longest monotonically
   increasing subsequence in a single channel's readings
3. **Edit Distance (Levenshtein)** — minimum insert/delete/replace operations
   to transform one signal string into another (used in error detection)


## Step 1 — Generate signal sequences

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_A  = random.randint(10, 16)
N_B  = random.randint(10, 16)
N_LIS = random.randint(14, 20)
ALPHABET = list("ABCDEFGH")

SEQ_A  = [random.choice(ALPHABET) for _ in range(N_A)]
SEQ_B  = [random.choice(ALPHABET) for _ in range(N_B)]
LIS_SEQ = [random.randint(1, 20) for _ in range(N_LIS)]

# For edit distance: short signal strings (easier to trace)
N_ED  = random.randint(7, 11)
SIG_X = "".join(random.choices("ABCDE", k=N_ED))
SIG_Y = "".join(random.choices("ABCDE", k=N_ED - random.randint(1,3)))

print(f"Signal sequences:")
print(f"  Channel A (n={N_A}) : {SEQ_A}")
print(f"  Channel B (n={N_B}) : {SEQ_B}")
print(f"  LIS input (n={N_LIS}) : {LIS_SEQ}")
print(f"  Signal X : {SIG_X}  (len={len(SIG_X)})")
print(f"  Signal Y : {SIG_Y}  (len={len(SIG_Y)})")


## Step 2 — Longest Common Subsequence

In [ ]:
def lcs(a, b):
    """
    Longest Common Subsequence using 2D DP table.

    dp[i][j] = length of LCS of a[:i] and b[:j].
    Base: dp[0][j] = dp[i][0] = 0  (empty prefix has no common subsequence)
    Recurrence:
      if a[i-1] == b[j-1]: dp[i][j] = dp[i-1][j-1] + 1   (match — extend LCS)
      else:                 dp[i][j] = max(dp[i-1][j],     (skip a[i-1])
                                           dp[i][j-1])     (skip b[j-1])
    Answer: dp[len(a)][len(b)]

    Backtracking: follow the decisions made (match → go diagonal, else go max direction).
    O(m*n) time and space.
    """
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    # Backtrack to reconstruct LCS
    result = []
    i, j = m, n
    while i > 0 and j > 0:
        if a[i-1] == b[j-1]:
            result.append(a[i-1]); i -= 1; j -= 1
        elif dp[i-1][j] > dp[i][j-1]:
            i -= 1
        else:
            j -= 1
    result.reverse()
    return dp[m][n], result, dp

lcs_len, lcs_seq, lcs_dp = lcs(SEQ_A, SEQ_B)

print(f"Longest Common Subsequence:")
print(f"  Channel A : {SEQ_A}")
print(f"  Channel B : {SEQ_B}")
print()
print(f"  DP table (rows=A, cols=B):")
header = "         " + "  ".join(f"{c:>2}" for c in SEQ_B)
print(f"  {header}")
print(f"  {'':>5}  " + "  ".join(f"{v:2}" for v in lcs_dp[0]))
for i, row in enumerate(lcs_dp[1:], 1):
    print(f"  {SEQ_A[i-1]:>4}   " + "  ".join(f"{v:2}" for v in row))
print()
print(f"  LCS length : {lcs_len}")
print(f"  LCS        : {lcs_seq}")
# Verify
def is_subseq(sub, seq):
    it = iter(seq)
    return all(c in it for c in sub)
print(f"  Is subseq of A: {'✅' if is_subseq(lcs_seq, SEQ_A) else '❌'}")
print(f"  Is subseq of B: {'✅' if is_subseq(lcs_seq, SEQ_B) else '❌'}")


## Step 3 — Longest Increasing Subsequence

In [ ]:
def lis_dp(nums):
    """
    Longest Increasing Subsequence — O(n log n) patience sort variant.

    Maintain a list `tails` where tails[i] = smallest tail element of any
    increasing subsequence of length i+1 found so far.

    For each number x:
      - Binary search for the leftmost position in tails where tails[pos] >= x
      - Replace tails[pos] with x (or extend tails if x is larger than all)

    tails is always sorted → binary search works.
    The LENGTH of tails at the end = LIS length.
    To reconstruct the actual subsequence, track predecessors.

    O(n log n) time, O(n) space.
    """
    import bisect
    n = len(nums)
    tails     = []    # smallest tail of increasing subseq of each length
    parent    = [-1] * n
    tail_idx  = [0]  * n   # index of element that set tails[k]

    for i, x in enumerate(nums):
        pos = bisect.bisect_left(tails, x)
        if pos == len(tails):
            tails.append(x)
        else:
            tails[pos] = x
        tail_idx[pos] = i
        parent[i] = tail_idx[pos - 1] if pos > 0 else -1

    # Reconstruct LIS by backtracking through parent array
    lis_length = len(tails)
    idx = tail_idx[lis_length - 1]
    subseq = []
    while idx != -1:
        subseq.append(nums[idx])
        idx = parent[idx]
    subseq.reverse()
    return lis_length, subseq, tails[:]

lis_len, lis_subseq, lis_tails = lis_dp(LIS_SEQ)

print(f"Longest Increasing Subsequence (n={N_LIS}):")
print(f"  Input    : {LIS_SEQ}")
print()
print(f"  LIS length   : {lis_len}")
print(f"  LIS          : {lis_subseq}")
print(f"  Final tails  : {lis_tails}")
print()
# Verify
is_inc = all(lis_subseq[i] < lis_subseq[i+1] for i in range(len(lis_subseq)-1))
is_sub = is_subseq(lis_subseq, LIS_SEQ)
print(f"  Is subsequence of input : {'✅' if is_sub else '❌'}")
print(f"  Is strictly increasing  : {'✅' if is_inc else '❌'}")
print()

# Show patience sort steps (first 8)
import bisect
tails_trace = []
print(f"  Patience sort trace (first 8 elements):")
tails_t = []
for i, x in enumerate(LIS_SEQ[:8]):
    pos = bisect.bisect_left(tails_t, x)
    action = "extend" if pos == len(tails_t) else f"replace pos {pos}"
    if pos == len(tails_t): tails_t.append(x)
    else: tails_t[pos] = x
    print(f"    x={x:3d}  → {action:<18}  tails={tails_t}")


## Step 4 — Edit Distance (Levenshtein)

In [ ]:
def edit_distance(s, t):
    """
    Minimum edit distance (Levenshtein): fewest insert/delete/replace
    operations to transform string s into string t.

    dp[i][j] = edit distance between s[:i] and t[:j].
    Base: dp[0][j] = j  (delete all of t[:j])
          dp[i][0] = i  (insert all of s[:i])
    Recurrence:
      if s[i-1] == t[j-1]: dp[i][j] = dp[i-1][j-1]          (no cost — match)
      else: dp[i][j] = 1 + min(dp[i-1][j],    # delete s[i-1]
                                dp[i][j-1],    # insert t[j-1]
                                dp[i-1][j-1])  # replace s[i-1] with t[j-1]
    Answer: dp[len(s)][len(t)]
    O(m*n) time and space.
    """
    m, n = len(s), len(t)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            if s[i-1] == t[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    # Backtrack to reconstruct operations
    ops = []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and s[i-1] == t[j-1]:
            ops.append(f"match   {s[i-1]}")
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            ops.append(f"replace {s[i-1]} → {t[j-1]}")
            i -= 1; j -= 1
        elif j > 0 and dp[i][j] == dp[i][j-1] + 1:
            ops.append(f"insert  {t[j-1]}")
            j -= 1
        else:
            ops.append(f"delete  {s[i-1]}")
            i -= 1
    ops.reverse()
    return dp[m][n], dp, ops

ed_dist, ed_dp, ed_ops = edit_distance(SIG_X, SIG_Y)

print(f"Edit Distance:")
print(f"  Signal X : {SIG_X}  (len={len(SIG_X)})")
print(f"  Signal Y : {SIG_Y}  (len={len(SIG_Y)})")
print()
print(f"  DP table (rows=X, cols=Y):")
header = "       " + "  ".join(f"{c:>2}" for c in " " + SIG_Y)
print(f"  {header}")
for i, row in enumerate(ed_dp):
    lbl = SIG_X[i-1] if i > 0 else " "
    print(f"  {lbl:>3}  " + "  ".join(f"{v:2}" for v in row))
print()
print(f"  Edit distance : {ed_dist}")
print()
print(f"  Operations to transform X → Y ({len(ed_ops)} steps):")
for op in ed_ops:
    print(f"    {op}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " SIGNAL SEQUENCE ANALYSER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  LCS (LONGEST COMMON SUBSEQUENCE)" + " "*(W-34) + "║")
print(f"║  {'Channel A length':<28}: {N_A:<26}║")
print(f"║  {'Channel B length':<28}: {N_B:<26}║")
print(f"║  {'LCS length':<28}: {lcs_len:<26}║")
print(f"║  {'LCS':<28}: {str(lcs_seq):<26}║")
print("╠" + "═"*W + "╣")
print("║  LIS (LONGEST INCREASING SUBSEQUENCE)" + " "*(W-38) + "║")
print(f"║  {'Input length':<28}: {N_LIS:<26}║")
print(f"║  {'LIS length':<28}: {lis_len:<26}║")
print(f"║  {'LIS':<28}: {str(lis_subseq):<26}║")
print(f"║  {'Strictly increasing':<28}: {'YES ✅' if is_inc else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  EDIT DISTANCE (LEVENSHTEIN)" + " "*(W-29) + "║")
print(f"║  {'Signal X':<28}: {SIG_X:<26}║")
print(f"║  {'Signal Y':<28}: {SIG_Y:<26}║")
print(f"║  {'Edit distance':<28}: {ed_dist:<26}║")
print(f"║  {'Operations':<28}: {len(ed_ops):<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about DP in this context?

*Your answer here:*

---

**Q2 — Memoization vs Tabulation — which did you find clearer, and why?**
Describe the trade-offs between top-down (memoization) and bottom-up (tabulation) for one of the problems in this project.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
